# Letterboxd → TMDB scrape (v4, multiprocessing, fixed)

The previous v3 broke because `multiprocess` + `dill` couldn't ship the helper function chain (`scrape_user_films` → `get_with_retry` → `session`) to spawned workers on Windows. Fix: this notebook auto-writes a `lbx_helpers.py` module on first run, and we import from it. Spawned workers can then re-import the module cleanly.

**You don't need to edit `lbx_helpers.py` by hand.** Just run the cells in order.


In [ ]:
# ===== 1. Install deps =====
import sys, subprocess
for pkg in ["curl_cffi", "beautifulsoup4", "pandas", "tqdm", "multiprocess"]:
    mod = {"beautifulsoup4": "bs4"}.get(pkg, pkg)
    try: __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("deps OK")


In [ ]:
# ===== 2. Write the helpers module to disk =====
# Workers spawned by multiprocess.Pool will `import` this file directly,
# which avoids the "function pickled but globals missing" problem.
HELPERS_SRC = '"""Auto-written by the v4 notebook. Don\'t edit by hand.\nIf you do edit, restart the kernel before re-running."""\nfrom __future__ import annotations\nimport re, time\nfrom typing import Optional\nfrom curl_cffi import requests\nfrom bs4 import BeautifulSoup\n\nLBX_BASE  = "https://letterboxd.com"\nTMDB_BASE = "https://api.themoviedb.org/3"\n\n_SESSION: Optional[requests.Session] = None\ndef session():\n    global _SESSION\n    if _SESSION is None:\n        _SESSION = requests.Session(impersonate="chrome124")\n        _SESSION.headers.update({\n            "Accept-Language": "en-US,en;q=0.9",\n            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",\n        })\n    return _SESSION\n\ndef get_with_retry(url, *, max_retries=5, timeout=20):\n    backoff = 1.0\n    for _ in range(max_retries):\n        try:\n            r = session().get(url, timeout=timeout)\n        except Exception:\n            time.sleep(backoff); backoff *= 2; continue\n        if r.status_code == 200:\n            if len(r.text) < 8000 and "Just a moment" in r.text:\n                time.sleep(backoff); backoff *= 2; continue\n            return r\n        if r.status_code == 404:\n            return r\n        if r.status_code in (403, 429, 500, 502, 503, 504):\n            time.sleep(backoff); backoff *= 2; continue\n        return r\n    return None\n\nRATING_RE    = re.compile(r"rated-(\\d+)")\nTMDB_LINK_RE = re.compile(r"themoviedb\\.org/movie/(\\d+)")\n\ndef scrape_user_films(username, max_pages=1000):\n    out, seen = [], set()\n    page = 1\n    while page <= max_pages:\n        r = get_with_retry(f"{LBX_BASE}/{username}/films/page/{page}/")\n        if r is None or r.status_code != 200:\n            break\n        soup = BeautifulSoup(r.text, "html.parser")\n        items = (soup.select("li.poster-container")\n                 or soup.select("li.griditem")\n                 or soup.select("li[data-owner-rating], li[data-film-id]"))\n        if not items:\n            break\n        new = 0\n        for item in items:\n            poster = item.select_one(\n                "div[data-film-slug], div[data-target-link^=\'/film/\'], "\n                "div.film-poster, div.really-lazy-load"\n            )\n            if poster is None and item.get("data-film-slug"):\n                poster = item\n            if poster is None:\n                continue\n            slug = poster.get("data-film-slug")\n            if not slug:\n                target = poster.get("data-target-link") or ""\n                if target.startswith("/film/"):\n                    slug = target.strip("/").split("/")[-1]\n            if not slug or slug in seen:\n                continue\n            seen.add(slug)\n            img = poster.select_one("img")\n            title = (img.get("alt") if img else None) or slug.replace("-", " ").title()\n            rating = None\n            owner = item.get("data-owner-rating")\n            if owner and owner not in ("0", ""):\n                try: rating = int(owner) / 2.0\n                except (TypeError, ValueError): pass\n            if rating is None:\n                rs = item.select_one("span.rating")\n                if rs:\n                    m = RATING_RE.search(" ".join(rs.get("class", [])))\n                    if m: rating = int(m.group(1)) / 2.0\n            out.append({"username": username, "slug": slug, "title": title, "rating": rating})\n            new += 1\n        if new == 0 or not soup.select_one("div.paginate-nextprev a.next"):\n            break\n        page += 1\n    return out\n\ndef resolve_tmdb_id(slug):\n    r = get_with_retry(f"{LBX_BASE}/film/{slug}/")\n    if r is None or r.status_code != 200:\n        return {"slug": slug, "tmdb_id": None, "title": None}\n    m = TMDB_LINK_RE.search(r.text)\n    tmdb_id = int(m.group(1)) if m else None\n    soup = BeautifulSoup(r.text, "html.parser")\n    title_tag = soup.select_one("h1.headline-1, h1.filmtitle, h1")\n    title = title_tag.get_text(strip=True) if title_tag else None\n    return {"slug": slug, "tmdb_id": tmdb_id, "title": title}\n\ndef fetch_tmdb(args):\n    tmdb_id, api_key = args\n    url = f"{TMDB_BASE}/movie/{tmdb_id}?api_key={api_key}&language=en-US"\n    backoff = 0.5\n    for _ in range(5):\n        try:\n            r = session().get(url, timeout=20)\n        except Exception:\n            time.sleep(backoff); backoff *= 2; continue\n        if r.status_code == 200:\n            d = r.json()\n            return {"tmdb_id": tmdb_id,\n                    "title": d.get("title") or d.get("original_title"),\n                    "genres": [g["name"] for g in d.get("genres", [])]}\n        if r.status_code == 429:\n            time.sleep(float(r.headers.get("Retry-After", backoff))); backoff *= 2; continue\n        if r.status_code in (500, 502, 503, 504):\n            time.sleep(backoff); backoff *= 2; continue\n        return {"tmdb_id": tmdb_id, "title": None, "genres": []}\n    return {"tmdb_id": tmdb_id, "title": None, "genres": []}\n\ndef phase1_worker(username):\n    """Re-raises errors instead of silently swallowing — Pool will surface them."""\n    return username, scrape_user_films(username)\n'
with open("lbx_helpers.py", "w", encoding="utf-8") as fh:
    fh.write(HELPERS_SRC)

# Force re-import in case it was edited
import importlib, sys, os
sys.path.insert(0, os.getcwd())
if "lbx_helpers" in sys.modules:
    importlib.reload(sys.modules["lbx_helpers"])
import lbx_helpers
from lbx_helpers import scrape_user_films, resolve_tmdb_id, fetch_tmdb, phase1_worker
print("helpers loaded from", lbx_helpers.__file__)


In [ ]:
# ===== 3. Configuration =====
TMDB_API_KEY  = "PASTE_YOUR_TMDB_V3_API_KEY_HERE"
USERNAMES_CSV = "usernames.csv"

WORKERS_LBX   = 24
WORKERS_TMDB  = 40

MAX_USERS     = None
WIPE_AND_RESTART = False

from pathlib import Path
PHASE1 = Path("phase1.jsonl")
PHASE2 = Path("phase2.jsonl")
PHASE3 = Path("phase3.jsonl")
FINAL  = Path("letterboxd_reviews.csv")

if WIPE_AND_RESTART:
    for p in (PHASE1, PHASE2, PHASE3, FINAL):
        if p.exists():
            p.unlink(); print("removed", p)
print("config OK; workers =", WORKERS_LBX, "/", WORKERS_TMDB)


In [ ]:
# ===== 4. Imports =====
import json
from multiprocess import Pool
import pandas as pd
from tqdm.auto import tqdm


## Connectivity check (in the kernel — no Pool yet)

Must succeed before going further. If this fails the rest will fail.

In [ ]:
_films = scrape_user_films("schaffrillas", max_pages=1)
print(f"got {len(_films)} films from schaffrillas page 1")
print("sample:", _films[:3])
assert len(_films) > 0, "Letterboxd scrape returned 0 films"

_g = fetch_tmdb((496243, TMDB_API_KEY))
print("TMDB sample:", _g)
assert _g["genres"], "TMDB returned no genres - check TMDB_API_KEY"


## Pool sanity check

Spins up 2 workers and runs `phase1_worker` once on a known user. If this fails with `NameError`, the helpers module didn't import cleanly in the worker.


In [ ]:
with Pool(processes=2) as _p:
    out = list(_p.imap_unordered(phase1_worker, ["schaffrillas"]))
print("pool returned:", len(out), "result(s)")
print("first user, n_films:", out[0][0], len(out[0][1]))
assert len(out[0][1]) > 0, "Pool worker returned 0 films - check that lbx_helpers.py is on the worker path"


## Load usernames

In [ ]:
usernames = (
    pd.read_csv(USERNAMES_CSV)["username"]
    .dropna().astype(str).str.strip().str.lower()
    .unique().tolist()
)
if MAX_USERS:
    usernames = usernames[:MAX_USERS]
print(f"{len(usernames):,} usernames queued")


## Phase 1 — scrape every user's films

In [ ]:
def phase1_done():
    done = set()
    if not PHASE1.exists(): return done
    with PHASE1.open() as fh:
        for line in fh:
            try: rec = json.loads(line)
            except: continue
            if rec.get("films"): done.add(rec["username"])
    return done

def run_phase1(usernames, workers):
    done = phase1_done()
    todo = [u for u in usernames if u not in done]
    print(f"Phase 1: {len(done):,} have films, {len(todo):,} to scrape")
    if not todo: return
    fh = PHASE1.open("a", buffering=1); counts = []
    try:
        with Pool(processes=workers) as pool:
            for username, films in tqdm(
                pool.imap_unordered(phase1_worker, todo, chunksize=1),
                total=len(todo), desc="Phase 1",
            ):
                fh.write(json.dumps({"username": username, "films": films}) + "\n")
                counts.append(len(films))
    finally:
        fh.close()
    if counts: print(f"avg films/user: {sum(counts)/len(counts):.1f}")

run_phase1(usernames, WORKERS_LBX)


## Aggregate Phase 1

In [ ]:
rows = []
with PHASE1.open() as fh:
    for line in fh:
        try: rec = json.loads(line)
        except: continue
        rows.extend(rec.get("films", []))
films_df = pd.DataFrame(rows)
print(f"{len(films_df):,} (user, film) rows across {films_df['username'].nunique():,} users")
unique_slugs = films_df["slug"].dropna().unique().tolist()
print(f"{len(unique_slugs):,} unique slugs to resolve")
films_df.head()


## Phase 2 — slug → TMDB id

In [ ]:
def phase2_done():
    done = set()
    if not PHASE2.exists(): return done
    with PHASE2.open() as fh:
        for line in fh:
            try: done.add(json.loads(line)["slug"])
            except: continue
    return done

def run_phase2(slugs, workers):
    done = phase2_done()
    todo = [s for s in slugs if s not in done]
    print(f"Phase 2: {len(done):,} resolved, {len(todo):,} to go")
    if not todo: return
    fh = PHASE2.open("a", buffering=1)
    try:
        with Pool(processes=workers) as pool:
            for rec in tqdm(
                pool.imap_unordered(resolve_tmdb_id, todo, chunksize=8),
                total=len(todo), desc="Phase 2",
            ):
                fh.write(json.dumps(rec) + "\n")
    finally:
        fh.close()

run_phase2(unique_slugs, WORKERS_LBX)


## Aggregate Phase 2

In [ ]:
slug_rows = []
with PHASE2.open() as fh:
    for line in fh:
        try: slug_rows.append(json.loads(line))
        except: continue
slug_df = pd.DataFrame(slug_rows)
slug_df["tmdb_id"] = pd.to_numeric(slug_df["tmdb_id"], errors="coerce").astype("Int64")
print(f"{slug_df['tmdb_id'].notna().sum():,} / {len(slug_df):,} slugs got a TMDB id")
unique_tmdb_ids = slug_df["tmdb_id"].dropna().astype(int).unique().tolist()
print(f"{len(unique_tmdb_ids):,} unique TMDB ids")


## Phase 3 — TMDB genres

In [ ]:
def phase3_done():
    done = set()
    if not PHASE3.exists(): return done
    with PHASE3.open() as fh:
        for line in fh:
            try: done.add(int(json.loads(line)["tmdb_id"]))
            except: continue
    return done

def run_phase3(tmdb_ids, api_key, workers):
    if not api_key or api_key.startswith("PASTE_"):
        raise RuntimeError("Set TMDB_API_KEY in the config cell first.")
    done = phase3_done()
    todo = [t for t in tmdb_ids if t not in done]
    print(f"Phase 3: {len(done):,} fetched, {len(todo):,} to go")
    if not todo: return
    args = [(t, api_key) for t in todo]
    fh = PHASE3.open("a", buffering=1)
    try:
        with Pool(processes=workers) as pool:
            for rec in tqdm(
                pool.imap_unordered(fetch_tmdb, args, chunksize=16),
                total=len(args), desc="Phase 3",
            ):
                fh.write(json.dumps(rec) + "\n")
    finally:
        fh.close()

run_phase3(unique_tmdb_ids, TMDB_API_KEY, WORKERS_TMDB)


## Final merge → CSV

In [ ]:
genre_rows = []
with PHASE3.open() as fh:
    for line in fh:
        try: genre_rows.append(json.loads(line))
        except: continue
genre_df = pd.DataFrame(genre_rows)
genre_df["tmdb_id"] = genre_df["tmdb_id"].astype("Int64")

merged = (
    films_df.merge(slug_df[["slug", "tmdb_id"]], on="slug", how="left")
            .merge(genre_df[["tmdb_id", "title", "genres"]], on="tmdb_id", how="left",
                   suffixes=("", "_tmdb"))
)
merged["movie_title"] = merged["title_tmdb"].fillna(merged["title"])

def split_genres(gs):
    gs = gs if isinstance(gs, list) else []
    gs = (gs + [None] * 6)[:6]
    return pd.Series(gs, index=["genre", "subgenre_1", "subgenre_2", "subgenre_3", "subgenre_4", "subgenre_5"])

genre_cols = merged["genres"].apply(split_genres)
final = pd.concat([merged[["username", "tmdb_id", "movie_title", "rating"]], genre_cols], axis=1)

final.to_csv(FINAL, index=False)
print(f"Wrote {FINAL.resolve()}  ({len(final):,} rows, {FINAL.stat().st_size/1024/1024:.1f} MB)")
final.head()


In [ ]:
print("Rows                :", f"{len(final):,}")
print("Unique users        :", f"{final['username'].nunique():,}")
print("Unique movies (tmdb):", f"{final['tmdb_id'].nunique():,}")
print("With rating         :", f"{final['rating'].notna().sum():,}")
print("With tmdb_id        :", f"{final['tmdb_id'].notna().sum():,}")
print("With at least 1 genre:", f"{final['genre'].notna().sum():,}")
print()
print("Top genres:")
print(final["genre"].value_counts().head(10))
